In [ ]:
import datasets
import numpy as np
import os
from PIL import Image
from datasets import Image as Image_ds
from datasets import ClassLabel, Features
import pandas as pd 
import torch
from tqdm import tqdm
from collections import Counter
import os
from torch import optim, nn, utils, Tensor
from torchvision.transforms import ToTensor
import lightning as L

In [2]:
ds = datasets.load_from_disk(os.path.join('..', 'data', 'wikiart_filtered_remapped_FINAL'))

In [20]:
ds.features['artist']

ClassLabel(names=['boris-kustodiev', 'camille-pissarro', 'childe-hassam', 'claude-monet', 'edgar-degas', 'eugene-boudin', 'gustave-dore', 'ilya-repin', 'ivan-aivazovsky', 'ivan-shishkin', 'john-singer-sargent', 'marc-chagall', 'martiros-saryan', 'nicholas-roerich', 'pablo-picasso', 'paul-cezanne', 'pierre-auguste-renoir', 'pyotr-konchalovsky', 'raphael-kirchner', 'rembrandt', 'salvador-dali', 'vincent-van-gogh', 'hieronymus-bosch', 'leonardo-da-vinci', 'albrecht-durer', 'edouard-cortes', 'sam-francis', 'juan-gris', 'lucas-cranach-the-elder', 'paul-gauguin', 'konstantin-makovsky', 'egon-schiele', 'thomas-eakins', 'francisco-goya', 'edvard-munch', 'henri-matisse', 'fra-angelico', 'maxime-maufra', 'jan-matejko', 'mstislav-dobuzhinsky', 'alfred-sisley', 'mary-cassatt', 'gustave-loiseau', 'fernando-botero', 'zinaida-serebriakova', 'georges-seurat', 'isaac-levitan', 'joaquã\xadn-sorolla', 'berthe-morisot', 'arkhip-kuindzhi', 'james-tissot', 'vasily-polenov', 'valentin-serov', 'pietro-perugin

In [21]:
sorted(list(set(ds[f"artist_str"])))

['albrecht-durer',
 'aleksey-savrasov',
 'alfred-sisley',
 'amedeo-modigliani',
 'anthony-van-dyck',
 'antoine-blanchard',
 'arkhip-kuindzhi',
 'aubrey-beardsley',
 'bartolome-esteban-murillo',
 'berthe-morisot',
 'boris-kustodiev',
 'camille-corot',
 'camille-pissarro',
 'childe-hassam',
 'claude-monet',
 'dante-gabriel-rossetti',
 'david-burliuk',
 'edgar-degas',
 'edouard-cortes',
 'edouard-manet',
 'edvard-munch',
 'egon-schiele',
 'el-greco',
 'ernst-ludwig-kirchner',
 'eugene-boudin',
 'felix-vallotton',
 'ferdinand-hodler',
 'fernand-leger',
 'fernando-botero',
 'fra-angelico',
 'francisco-goya',
 'frans-hals',
 'gene-davis',
 'georges-braque',
 'georges-seurat',
 'giovanni-boldini',
 'gustave-caillebotte',
 'gustave-courbet',
 'gustave-dore',
 'gustave-loiseau',
 'hans-holbein-the-younger',
 'hans-memling',
 'henri-de-toulouse-lautrec',
 'henri-fantin-latour',
 'henri-martin',
 'henri-matisse',
 'hieronymus-bosch',
 'ilya-mashkov',
 'ilya-repin',
 'isaac-levitan',
 'ivan-aivazo

In [30]:
def remap_features(ds_original, ds_filtered, label):

    original_feature = ds_original.features[label] # the ClassLabel feature
    original_names = original_feature.names

    # classes(names) in new subclassification dataset:
    used_class_names = sorted(list(set(ds_filtered[f"{label}_str"])))
    new_class_label = ClassLabel(names=used_class_names)

    # set up function to remap from str -> int for new ClassLabels
    def remap_labels(example):
        example[label] = new_class_label.str2int(example[f"{label}_str"])
        return example
    
    # use map to remap classlabels
    ds_filtered = ds_filtered.map(remap_labels)

    # recast the class label feature to new labels
    new_features = ds_filtered.features.copy()
    new_features[label] = new_class_label
    ds_filtered = ds_filtered.cast(new_features)

    return ds_filtered

In [ ]:
# write function that allows to filter data based on input parameters:

def filter_data(ds, label, subclassification_task, seed):
    ds = ds.add_column('old_indices', range(len(ds)))

    # find the rows that matches the subclassification task
    subclass_indices = [idx for idx, a in enumerate(ds[f'{label}_str']) if a in subclassification_task]
    ds_subset = ds.select(subclass_indices)

    # remap labels to fit to new number of classes for subclassification task
    ds_subset = remap_features(ds, ds_subset, label)

    # split into train, val and test: 
    ds_split = ds_subset.train_test_split(test_size=0.2, seed=seed, stratify_by_column = label)
    ds_train = ds_split['train']
    ds_test = ds_split['test']

    # split test data into test and validation
    ds_test_split = ds_test.train_test_split(test_size=0.5, seed=seed, stratify_by_column = label)
    ds_val = ds_test_split['train']
    ds_test = ds_test_split['test']

    ds_splits = {
            'train': ds_train,
            'test': ds_test,
            'val': ds_val}

    return ds_splits

In [41]:
rembrandt_vs_rubens = ['rembrandt', 'peter-paul-rubens']
ds_splits = filter_data(ds, 'artist', rembrandt_vs_rubens, 2830)

### Test with PyTorch lightning

In [ ]:
# figure out how to make these into parameters instead
hidden_layer_size = 200
label = 'artist'
inp_size = 1014 # dims of embedding
dropout_p = 0.3
device = 'cpu'

num_classes = ds_splits['train'].features[label].num_classes

model = nn.Sequential(
        nn.Linear(in_features=inp_size, out_features=hidden_layer_size),
        nn.ReLU(),
        nn.Dropout(p=dropout_p),
        nn.Linear(in_features=hidden_layer_size, out_features=num_classes)
            ).to(device)

In [ ]:
# what exactly is this doing
L.seed_everything(1121218)

In [ ]:
def define_class_weights(ds_splits, label):
    y_tensor = torch.tensor(ds_splits['train'][label])
    class_counts = torch.bincount(y_tensor)
    class_weights = 1.0 / class_counts.float() # weight the loss inversely proportional to class frequency
    class_weights /= class_weights.sum() # normalize weights so they sum to one

    return class_weights

In [ ]:
# define a LightningModule

# define the model with a build in training loop, validation loop, test loop + optimizer
class SubclassModel(L.LightningModule):
    def __init__(self, model, class_weights, lr, weight_decay): # options to set some default parameters here

        # not really sure what this does:
        super().__init__()

        self.model = model
        self.lr = lr 
        self.weight_decay = weight_decay 

        # don't really get this either
        self.register_buffer('class_weights', class_weights)
        self.loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)

    def training_step(self, batch, batch_idx):
        x, y = batch 
        y_hat = self.model(x)
        loss = self.loss_fn(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()

        # log the training loss and accuracy
        # Log the loss at each training step and epoch, create a progress bar
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True) # logged per-epoch level
        self.log("train_acc", acc, on_step=True, on_epoch=True, prog_bar=True, logger=True)

        return loss 
    
    def validation_step(self, batch, batch_idx):
        x, y = batch 
        y_hat = self.model(x)
        loss = self.loss_fn(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log('val_loss', loss)
        self.log('val_acc', acc)
    
    def test_step(self, batch, batch_idx):
        x, y = batch 
        y_hat = self.model(x)
        loss = self.loss_fn(y_hat, y)
        acc = (y_hat.argmax(1) == y).float().mean()
        self.log('test_loss', loss)
        self.log('test_acc', acc) 

    def configure_optimizers(self, lr, weight_decay):
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=decay)

        

In [39]:
nn_model = SubclassModel(model)

NameError: name 'model' is not defined